# Processing and Qdrant indexing pipeline

## Goal
Run production chunking, tagging, embedding, BM25, and Qdrant indexing without a model download or server.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT))

## Setup

A deterministic encoder and Qdrant's in-memory client make this repeatable.

In [2]:
import numpy as np
from qdrant_client import QdrantClient
from processing.chunker import SectionAwareChunker
from processing.embedder import Embedder
from processing.metadata_tagger import MetadataTagger
from processing.qdrant_indexer import QdrantIndexer
from processing.pipeline import ProcessingPipeline
class DiagnosticEncoder:
 def encode(self,texts,**kwargs): return np.array([[len(t),t.count('retrieval'),t.count('method')] for t in texts],dtype=float)

d:\data science\project\RAG-AI_Reasearch_Papers\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


## Steps

### 1. Define ingestion-compatible data

In [3]:
documents=[{'paper_id':'demo-paper','metadata':{'title':'Demo','authors':['Ada'],'year':2024,'categories':['cs.IR']},'sections':{'abstract':'A retrieval research abstract.','methodology':'The method combines dense and sparse retrieval.','results':'The method improves recall.'}}]

### 2. Run every processing stage

In [4]:
qdrant=QdrantIndexer(client=QdrantClient(':memory:'),collection_name='diagnostic')
pipeline=ProcessingPipeline(chunker=SectionAwareChunker(max_tokens=40,overlap_tokens=5),embedder=Embedder(model=DiagnosticEncoder()),metadata_tagger=MetadataTagger(),qdrant_indexer=qdrant)
result=pipeline.process_documents(documents)
{k:v for k,v in result.items() if k not in ('chunks','embedding_records')}

{'bm25_documents': 3, 'qdrant_points': 3}

### 3. Compare payloads and retrieval

In [5]:
[c.to_dict() for c in result['chunks']]

[{'chunk_id': '118892eb-32c3-5060-82db-120a46dfef28',
  'paper_id': 'demo-paper',
  'section': 'abstract',
  'text': 'A retrieval research abstract.',
  'start_char': 0,
  'end_char': 30,
  'metadata': {'title': 'Demo',
   'authors': ['Ada'],
   'year': 2024,
   'categories': ['cs.IR'],
   'paper_id': 'demo-paper',
   'section': 'abstract',
   'section_family': 'front'}},
 {'chunk_id': 'e4f2f7a3-94a6-5f86-a4c0-89097da4faa1',
  'paper_id': 'demo-paper',
  'section': 'methodology',
  'text': 'The method combines dense and sparse retrieval.',
  'start_char': 0,
  'end_char': 47,
  'metadata': {'title': 'Demo',
   'authors': ['Ada'],
   'year': 2024,
   'categories': ['cs.IR'],
   'paper_id': 'demo-paper',
   'section': 'methodology',
   'section_family': 'method'}},
 {'chunk_id': 'f5649601-4cf0-5869-a9dd-d70b598fbd21',
  'paper_id': 'demo-paper',
  'section': 'results',
  'text': 'The method improves recall.',
  'start_char': 0,
  'end_char': 27,
  'metadata': {'title': 'Demo',
   'author

In [6]:
pipeline.bm25_indexer.search('dense retrieval',top_k=2)

[{'chunk_id': 'e4f2f7a3-94a6-5f86-a4c0-89097da4faa1',
  'paper_id': 'demo-paper',
  'section': 'methodology',
  'text': 'The method combines dense and sparse retrieval.',
  'metadata': {'title': 'Demo',
   'authors': ['Ada'],
   'year': 2024,
   'categories': ['cs.IR'],
   'paper_id': 'demo-paper',
   'section': 'methodology',
   'section_family': 'method'},
  'score': 0.4870159548616436},
 {'chunk_id': '118892eb-32c3-5060-82db-120a46dfef28',
  'paper_id': 'demo-paper',
  'section': 'abstract',
  'text': 'A retrieval research abstract.',
  'metadata': {'title': 'Demo',
   'authors': ['Ada'],
   'year': 2024,
   'categories': ['cs.IR'],
   'paper_id': 'demo-paper',
   'section': 'abstract',
   'section_family': 'front'},
  'score': 0.0701683549129108}]

In [7]:
qdrant.search(DiagnosticEncoder().encode(['retrieval method'])[0].tolist(),limit=2)

[{'id': 'e4f2f7a3-94a6-5f86-a4c0-89097da4faa1',
  'score': 0.998313954968407,
  'payload': {'chunk_id': 'e4f2f7a3-94a6-5f86-a4c0-89097da4faa1',
   'paper_id': 'demo-paper',
   'section': 'methodology',
   'text': 'The method combines dense and sparse retrieval.',
   'metadata': {'title': 'Demo',
    'authors': ['Ada'],
    'year': 2024,
    'categories': ['cs.IR'],
    'paper_id': 'demo-paper',
    'section': 'methodology',
    'section_family': 'method'}}},
 {'id': 'f5649601-4cf0-5869-a9dd-d70b598fbd21',
  'score': 0.9977382409425053,
  'payload': {'chunk_id': 'f5649601-4cf0-5869-a9dd-d70b598fbd21',
   'paper_id': 'demo-paper',
   'section': 'results',
   'text': 'The method improves recall.',
   'metadata': {'title': 'Demo',
    'authors': ['Ada'],
    'year': 2024,
    'categories': ['cs.IR'],
    'paper_id': 'demo-paper',
    'section': 'results',
    'section_family': 'evidence'}}}]

## Checks

In [8]:
assert result['bm25_documents']==len(result['chunks'])==result['qdrant_points']
assert all('section_family' in c.metadata for c in result['chunks'])
print('All processing and indexing checks passed.')

All processing and indexing checks passed.


## Next Steps

Use a server-backed Qdrant client, `Embedder()`, and `ProcessingPipeline.process_paths` for production data.